# E5: Cross-Task Replication (Multiplication, Subtraction)

**Standalone experiment notebook.** This is E5 only (both E5a/multiplication and E5b/subtraction),
extracted from the full quantization/grokking project notebook, so it can be run as its own long,
crash-resilient job. It trains every seed to grokking (or gives up honestly), runs the full
post-training-quantization sweep, and exports a complete dataset for a downstream analysis notebook
(which will run E1/E6/E7, all statistics, and all figures) to load without recomputation.

`run_layerwise=False`, `run_fine_grained=False` for both tasks: the layer-group ablation and the
fine-grained level scan were being computed here and never used downstream -- H5's cross-task
analysis only ever reads PTQ rows -- confirmed and removed. `layerwise_rows`, `layerwise_fine_rows`,
and `fine_rows` will therefore always be empty for E5; this is expected, not a bug, and the export
step below skips saving empty files rather than cluttering the output. `mechanism_rows` is
different: it's cheap tensor arithmetic that isn't gated by either flag, so it will be non-empty
for any seed that grokked -- saved for completeness, even though only E1's mechanism data is
actually used downstream (H3's analysis).

**What this notebook does NOT do:** no evaluation, no statistics, no figures, no matplotlib at all
-- purely training plus a faithful export of the raw results.


## 1. Setup

In [1]:
import subprocess, sys

def pip_install(pkgs):
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
        print('Installed/verified:', ' '.join(pkgs))
    except Exception as e:
        print('pip install warning (continuing, likely already satisfied):', e)

pip_install(['scipy'])


Installed/verified: scipy


In [2]:
import os, sys, json, time, random, math, copy, zipfile
from datetime import datetime
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats

BASE_SEED = 0

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(BASE_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print('Could not enable full deterministic-algorithms mode (continuing):', e)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    print('WARNING: no CUDA GPU detected. Runtime > Change runtime type > T4 GPU is recommended.')
    print('The notebook will still run on CPU (much slower); consider QUICK_TEST = True below.')
else:
    print('GPU detected:', torch.cuda.get_device_name(0))

print('Environment info:')
print('  Python  :', sys.version.split()[0])
print('  PyTorch :', torch.__version__)
print('  NumPy   :', np.__version__)
print('  Pandas  :', pd.__version__)
print('  SciPy   :', __import__('scipy').__version__)
print('  CUDA available:', torch.cuda.is_available())
print('  Device  :', DEVICE)
print('  Run started:', datetime.now().isoformat())

EXPERIMENT_START_TIME = time.time()


GPU detected: Tesla T4
Environment info:
  Python  : 3.12.13
  PyTorch : 2.10.0+cu128
  NumPy   : 2.0.2
  Pandas  : 2.3.3
  SciPy   : 1.16.3
  CUDA available: True
  Device  : cuda
  Run started: 2026-07-18T18:12:27.072077


In [3]:
# ---------------- Environment detection (Colab / Kaggle / local) ----------------
# On Kaggle, relative paths like 'outputs/...' don't land in the directory Kaggle actually exposes
# for download (/kaggle/working/). Every path used below goes through OUTPUT_ROOT, set explicitly
# per environment, instead of a bare relative 'outputs/'.
def detect_runtime_environment():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    try:
        import google.colab  # noqa: F401
        return 'colab'
    except ImportError:
        return 'local'

RUNTIME_ENV = detect_runtime_environment()
if RUNTIME_ENV == 'kaggle':
    OUTPUT_ROOT = '/kaggle/working/outputs'
elif RUNTIME_ENV == 'colab':
    OUTPUT_ROOT = '/content/outputs'
else:
    OUTPUT_ROOT = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUTPUT_ROOT, exist_ok=True)
ZIP_PATH = os.path.join(os.path.dirname(OUTPUT_ROOT), 'e5_dataset.zip')
print('Runtime environment detected: {} | OUTPUT_ROOT = {} | ZIP_PATH = {}'.format(RUNTIME_ENV, OUTPUT_ROOT, ZIP_PATH))


Runtime environment detected: kaggle | OUTPUT_ROOT = /kaggle/working/outputs | ZIP_PATH = /kaggle/working/e5_dataset.zip


In [4]:
# ---------------- Checkpointing (Google Drive-backed, survives session crashes/disconnects) ----------------
# E5 (both tasks combined) is a long-running experiment -- exactly the case checkpointing exists
# for. A crashed or disconnected session loses everything in RAM; local Colab disk (/content/) is
# wiped along with the VM, so it does NOT protect against this. Google Drive does. Progress is
# saved after EVERY completed run (not just at the end), and on start, this checks for existing
# progress and resumes instead of recomputing -- so rerunning this notebook after a crash finishes
# in seconds for anything already done, and only computes what's missing.
import pickle

CLEAR_CHECKPOINTS = False  # set True to force a fully fresh run, ignoring any saved progress
CHECKPOINT_DIR = None

def setup_checkpointing():
    global CHECKPOINT_DIR
    if RUNTIME_ENV == 'colab':
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            CHECKPOINT_DIR = '/content/drive/MyDrive/quantization_experiment_checkpoints'
            os.makedirs(CHECKPOINT_DIR, exist_ok=True)
            print('Checkpointing ENABLED (Google Drive): {}'.format(CHECKPOINT_DIR))
            print('Progress survives a disconnected/crashed session -- rerunning this notebook resumes')
            print('instead of starting over.')
            return
        except Exception as e:
            print('WARNING: could not mount Google Drive ({}: {}).'.format(type(e).__name__, e))
            CHECKPOINT_DIR = '/content/checkpoints_local'
    elif RUNTIME_ENV == 'kaggle':
        CHECKPOINT_DIR = '/kaggle/working/checkpoints'
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        print('Checkpointing ENABLED (Kaggle working dir): {}'.format(CHECKPOINT_DIR))
        print('This survives an interrupted/crashed session within the same notebook edit session')
        print('(Kaggle preserves /kaggle/working during that time), but NOT starting a brand new')
        print('session from scratch -- there is no Kaggle equivalent of Drive-backed persistence.')
        return
    else:
        CHECKPOINT_DIR = os.path.join(os.getcwd(), 'checkpoints_local')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print('Falling back to LOCAL checkpointing at {} -- this will NOT survive a'.format(CHECKPOINT_DIR))
    print('disconnected or crashed session. To get real crash-resilience on Colab, allow Drive')
    print('access when prompted and re-run this cell.')
    if CLEAR_CHECKPOINTS:
        for f in os.listdir(CHECKPOINT_DIR):
            if f.endswith('.pkl'):
                os.remove(os.path.join(CHECKPOINT_DIR, f))
        print('CLEAR_CHECKPOINTS=True: removed all existing checkpoints, starting fully fresh.')

setup_checkpointing()

def save_checkpoint(name, data):
    if CHECKPOINT_DIR is None:
        return
    path = os.path.join(CHECKPOINT_DIR, '{}.pkl'.format(name))
    tmp_path = path + '.tmp'
    with open(tmp_path, 'wb') as f:
        pickle.dump(data, f)
    os.replace(tmp_path, path)  # atomic on POSIX: never leaves a half-written checkpoint on disk

def load_checkpoint(name):
    if CHECKPOINT_DIR is None:
        return None
    path = os.path.join(CHECKPOINT_DIR, '{}.pkl'.format(name))
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


def checkpointed_sweep(exp_name, combos, key_fields, run_fn, accumulator_names, save_every=1):
    """Run a list of (config-combo) experiments with automatic crash-resilient checkpointing.

    combos: list of dicts, each identifying one run's parameters (e.g. {'p': 97, 'seed': 0}).
    key_fields: which fields in each combo uniquely identify a run, for resume-skip logic.
    run_fn(combo) -> tuple of result pieces (dicts are appended, lists are extended, None is
        skipped), in the same order as `accumulator_names`.
    Saves progress after every `save_every` NEWLY completed runs (default: every single run --
    the per-run cost here is minutes, so the checkpoint-write overhead is negligible next to the
    risk of losing an hour of unattended compute to a disconnect).
    """
    ckpt = load_checkpoint(exp_name)
    if ckpt is not None:
        accumulators, completed_keys = ckpt['accumulators'], ckpt['completed_keys']
        print('[{}] Resumed from checkpoint: {} of {} planned runs already completed.'.format(
            exp_name, len(completed_keys), len(combos)))
    else:
        accumulators = {n: [] for n in accumulator_names}
        completed_keys = set()
        print('[{}] Starting fresh: {} runs planned.'.format(exp_name, len(combos)))

    n_new = 0
    for combo in combos:
        key = tuple(combo[f] for f in key_fields)
        if key in completed_keys:
            continue
        if time_remaining() < 120:
            print('[TIME BUDGET] Skipping remaining {} runs (checkpoint preserved).'.format(exp_name))
            break
        try:
            result = run_fn(combo)
        except Exception as e:
            print('  [RUN FAILED, skipping this combo, continuing] {}: {}: {}'.format(combo, type(e).__name__, e))
            continue
        for name, piece in zip(accumulator_names, result):
            if isinstance(piece, list):
                accumulators[name].extend(piece)
            elif piece is not None:
                accumulators[name].append(piece)
        completed_keys.add(key)
        n_new += 1
        if n_new % save_every == 0:
            save_checkpoint(exp_name, dict(accumulators=accumulators, completed_keys=completed_keys))
    save_checkpoint(exp_name, dict(accumulators=accumulators, completed_keys=completed_keys))
    print('[{}] done: {} new this session, {} total completed (elapsed: {:.1f} min)'.format(
        exp_name, n_new, len(completed_keys), (time.time() - EXPERIMENT_START_TIME) / 60))
    return accumulators


Checkpointing ENABLED (Kaggle working dir): /kaggle/working/checkpoints
This survives an interrupted/crashed session within the same notebook edit session
(Kaggle preserves /kaggle/working during that time), but NOT starting a brand new
session from scratch -- there is no Kaggle equivalent of Drive-backed persistence.


In [5]:
import subprocess as _sp
try:
    print(_sp.run(['nvidia-smi'], capture_output=True, text=True, timeout=10).stdout)
except Exception:
    print('nvidia-smi not available (CPU runtime, or Colab GPU not yet allocated).')


Sat Jul 18 18:12:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Method: Canonical Architecture

In [6]:
class TinyTransformer(nn.Module):
    """Canonical 1-layer, no-LayerNorm transformer (Nanda et al., 2023, Sec. 3), with a
    parameter-matched SwiGLU MLP (Sec 3.2) in place of the original plain ReLU-MLP. Input: 3-token
    sequence [a, b, =] (vocab_size = p+1); output read from the final ('=') position; causal
    attention masking. Quantization is applied post-hoc to a frozen state_dict (Sec 3.3), never
    inside this class or during training."""

    def __init__(self, p, d_model=128, n_heads=4, d_ff=341, seq_len=3):
        super().__init__()
        assert d_model % n_heads == 0
        self.p, self.d_model, self.n_heads, self.seq_len = p, d_model, n_heads, seq_len
        self.d_head = d_model // n_heads
        self.vocab_size = p + 1  # token id `p` is reserved for '='

        self.embed = nn.Embedding(self.vocab_size, d_model)
        self.pos_embed = nn.Embedding(seq_len, d_model)
        nn.init.normal_(self.embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)

        self.unembed = nn.Linear(d_model, p, bias=False)  # p classes: labels never include the '=' token

        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        self.register_buffer('causal_mask', causal_mask, persistent=False)

    def forward(self, tokens):
        # tokens: (B, seq_len) with values in [0, p] ('=' is token id p, appended by the dataset generator)
        x = self.embed(tokens) + self.pos_embed.weight[None, :, :]
        B, T, D = x.shape

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k, v = split_heads(self.W_Q(x)), split_heads(self.W_K(x)), split_heads(self.W_V(x))
        scores = (q @ k.transpose(-1, -2)) / (self.d_head ** 0.5)
        scores = scores.masked_fill(self.causal_mask[None, None, :T, :T], float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        z = (attn @ v).transpose(1, 2).reshape(B, T, D)
        x = x + self.W_O(z)

        h = F.silu(self.w_gate(x)) * self.w_up(x)
        x = x + self.w_down(h)

        return self.unembed(x[:, -1, :])  # final position is always '=' by construction


WEIGHT_PARAM_NAMES = ['embed.weight', 'pos_embed.weight', 'W_Q.weight', 'W_K.weight', 'W_V.weight',
                       'W_O.weight', 'w_gate.weight', 'w_up.weight', 'w_down.weight', 'unembed.weight']
LAYER_GROUPS = {
    'embedding': ['embed.weight', 'pos_embed.weight'],
    'attention': ['W_Q.weight', 'W_K.weight', 'W_V.weight', 'W_O.weight'],
    'mlp': ['w_gate.weight', 'w_up.weight', 'w_down.weight'],
    'unembed': ['unembed.weight'],
}
LAYER_GROUPS_FINE_MLP = {  # H4b: within-MLP decomposition (unused here -- run_layerwise=False for E5)
    'mlp_gate': ['w_gate.weight'],
    'mlp_up': ['w_up.weight'],
    'mlp_down': ['w_down.weight'],
}


In [7]:
def _self_test_architecture():
    torch.manual_seed(0)
    p_test = 13
    model = TinyTransformer(p_test, d_model=128, n_heads=4, d_ff=341, seq_len=3)
    n_params_mlp = sum(x.numel() for n, x in model.named_parameters() if n.startswith(('w_gate', 'w_up', 'w_down')))
    print('SwiGLU MLP params: {} (target: 130,944, {:.2%} of 2-matrix-ReLU baseline 131,072)'.format(
        n_params_mlp, n_params_mlp / 131072))
    assert abs(n_params_mlp - 130944) < 10

    tokens = torch.randint(0, p_test + 1, (5, 3))
    tokens[:, 2] = p_test  # '=' token
    logits = model(tokens)
    assert logits.shape == (5, p_test), 'expected logits shape (5, p), got {}'.format(logits.shape)
    print('Forward pass shape check passed:', logits.shape)

    # causal mask sanity: perturbing token 1 (b) should NOT change position-0 output (a can't see b);
    # perturbing token 0 (a) SHOULD change the '=' position output (through the causal path a -> =).
    model.eval()
    with torch.no_grad():
        t1 = tokens.clone()
        t2 = tokens.clone()
        t2[:, 1] = (t2[:, 1] + 1) % p_test  # change b only
        out1, out2 = model(t1), model(t2)
    assert not torch.allclose(out1, out2), 'changing b should change the output read from the = position'
    print('Causal-path sanity check passed: changing b changes the output at the = position, as expected.')

    print('WEIGHT_PARAM_NAMES ({} entries):'.format(len(WEIGHT_PARAM_NAMES)), WEIGHT_PARAM_NAMES)
    assert len(WEIGHT_PARAM_NAMES) == 10

_self_test_architecture()


SwiGLU MLP params: 130944 (target: 130,944, 99.90% of 2-matrix-ReLU baseline 131,072)
Forward pass shape check passed: torch.Size([5, 13])
Causal-path sanity check passed: changing b changes the output at the = position, as expected.
WEIGHT_PARAM_NAMES (10 entries): ['embed.weight', 'pos_embed.weight', 'W_Q.weight', 'W_K.weight', 'W_V.weight', 'W_O.weight', 'w_gate.weight', 'w_up.weight', 'w_down.weight', 'unembed.weight']


## 3. Post-Training Quantization Schemes

In [8]:
def fake_quantize_mantissa(x: torch.Tensor, bits: int, eps: float = 1e-12) -> torch.Tensor:
    """Simulated b-bit floating-point mantissa rounding (exponent untouched)."""
    if bits >= 23:
        return x
    sign = torch.sign(x)
    abs_x = torch.abs(x) + eps
    exponent = torch.floor(torch.log2(abs_x))
    mantissa = abs_x / (2.0 ** exponent)
    scale = float(2 ** bits)
    mantissa_q = torch.round(mantissa * scale) / scale
    x_q = sign * mantissa_q * (2.0 ** exponent)
    x_q = torch.where(torch.abs(x) < eps, torch.zeros_like(x), x_q)
    return x + (x_q - x).detach()


def int_quantize_nlevels(w: torch.Tensor, n_levels: int) -> torch.Tensor:
    """Uniform symmetric INT quantization parameterized by explicit level count."""
    if n_levels >= 2 ** 16:
        return w
    qmax = max((n_levels - 1) // 2, 1)
    scale = w.abs().max().item() / qmax
    if scale < 1e-12:
        return w
    return torch.round(w / scale).clamp(-qmax, qmax) * scale


def int_quantize(w: torch.Tensor, bits: int) -> torch.Tensor:
    if bits >= 23:
        return w
    return int_quantize_nlevels(w, 2 ** bits - 1)


def nf_quantize(w: torch.Tensor, bits: int) -> torch.Tensor:
    """Non-uniform, quantile-based post-training quantization."""
    if bits >= 23:
        return w
    n_levels = min(2 ** bits, 256)
    std = w.std().item()
    if std < 1e-12:
        return w
    probs = (torch.arange(n_levels, dtype=torch.float32, device=w.device) + 0.5) / n_levels
    levels = torch.distributions.Normal(0.0, 1.0).icdf(probs.clamp(1e-6, 1 - 1e-6)) * std
    levels, _ = torch.sort(levels)
    boundaries = (levels[:-1] + levels[1:]) / 2
    idx = torch.bucketize(w.reshape(-1), boundaries)
    return levels[idx].reshape(w.shape)


def quantize_state_dict(state_dict, bits, scheme='int', param_names=None):
    if param_names is None:
        param_names = WEIGHT_PARAM_NAMES
    new_sd = {}
    for name, w in state_dict.items():
        if name in param_names and bits < 23:
            if scheme == 'int':
                new_sd[name] = int_quantize(w, bits)
            elif scheme == 'mantissa':
                new_sd[name] = fake_quantize_mantissa(w, bits)
            elif scheme == 'nf':
                new_sd[name] = nf_quantize(w, bits)
            else:
                raise ValueError('unknown scheme: ' + scheme)
        else:
            new_sd[name] = w.clone()
    return new_sd


@torch.no_grad()
def evaluate_state_dict(model_template, state_dict, train_tokens, train_labels, test_tokens, test_labels):
    model_template.load_state_dict(state_dict)
    model_template.eval()
    train_acc = (model_template(train_tokens).argmax(-1) == train_labels).float().mean().item()
    test_acc = (model_template(test_tokens).argmax(-1) == test_labels).float().mean().item()
    return train_acc, test_acc


def _self_test_quantization():
    torch.manual_seed(0)
    x = torch.randn(20000)
    print('{:>5} | {:>10} | {:>10} | {:>10}'.format('bits', 'INT MAE', 'mant MAE', 'NF MAE'))
    prev_int_mae = None
    for b in [2, 3, 4, 6, 8, 12, 16, 23]:
        xi, xm, xn = int_quantize(x, b), fake_quantize_mantissa(x, b), nf_quantize(x, b)
        mae_i, mae_m, mae_n = (xi - x).abs().mean().item(), (xm - x).abs().mean().item(), (xn - x).abs().mean().item()
        print('{:>5} | {:>10.6f} | {:>10.6f} | {:>10.6f}'.format(b, mae_i, mae_m, mae_n))
        if prev_int_mae is not None:
            assert mae_i <= prev_int_mae + 1e-6
        prev_int_mae = mae_i
    assert int_quantize(x, 23).equal(x) and fake_quantize_mantissa(x, 23).equal(x) and nf_quantize(x, 23).equal(x)
    print('Quantization self-test passed (all three schemes).')

_self_test_quantization()


 bits |    INT MAE |   mant MAE |     NF MAE
    2 |   0.778290 |   0.036266 |   0.270745
    3 |   0.362198 |   0.018064 |   0.151236
    4 |   0.154796 |   0.009012 |   0.083188
    6 |   0.034758 |   0.002282 |   0.024322
    8 |   0.008560 |   0.000563 |   0.007035
   12 |   0.000532 |   0.000035 |   0.007035
   16 |   0.000033 |   0.000002 |   0.007035
   23 |   0.000000 |   0.000000 |   0.000000
Quantization self-test passed (all three schemes).


## 4. Dataset (Canonical 3-Token Format)

In [9]:
def _append_equals_token(tokens, p):
    eq_col = np.full((tokens.shape[0], 1), p, dtype=tokens.dtype)
    return np.concatenate([tokens, eq_col], axis=1)


def generate_modular_multiplication_dataset(p: int, train_frac: float, seed: int):
    vals = np.arange(1, p)
    a, b = np.meshgrid(vals, vals, indexing='ij')
    a, b = a.flatten(), b.flatten()
    labels = (a * b) % p
    tokens = _append_equals_token(np.stack([a, b], axis=1), p)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(tokens))
    n_train = int(train_frac * len(tokens))
    train_idx, test_idx = perm[:n_train], perm[n_train:]

    def to_t(idx):
        return (torch.tensor(tokens[idx], dtype=torch.long), torch.tensor(labels[idx], dtype=torch.long))

    train_tokens, train_labels = to_t(train_idx)
    test_tokens, test_labels = to_t(test_idx)
    return train_tokens, train_labels, test_tokens, test_labels


def generate_modular_subtraction_dataset(p: int, train_frac: float, seed: int):
    a, b = np.meshgrid(np.arange(p), np.arange(p), indexing='ij')
    a, b = a.flatten(), b.flatten()
    labels = (a - b) % p
    tokens = _append_equals_token(np.stack([a, b], axis=1), p)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(tokens))
    n_train = int(train_frac * len(tokens))
    train_idx, test_idx = perm[:n_train], perm[n_train:]

    def to_t(idx):
        return (torch.tensor(tokens[idx], dtype=torch.long), torch.tensor(labels[idx], dtype=torch.long))

    train_tokens, train_labels = to_t(train_idx)
    test_tokens, test_labels = to_t(test_idx)
    return train_tokens, train_labels, test_tokens, test_labels


def generate_dataset(task, p, train_frac, seed):
    if task == 'multiplication':
        return generate_modular_multiplication_dataset(p, train_frac, seed)
    elif task == 'subtraction':
        return generate_modular_subtraction_dataset(p, train_frac, seed)
    raise ValueError('this standalone notebook only runs E5 (multiplication, subtraction); got task=' + task)


for _task in ['multiplication', 'subtraction']:
    _tt, _tl, _et, _el = generate_dataset(_task, 97, 0.4, 0)
    assert _tt.shape[1] == 3, 'expected 3-token sequences, got width {}'.format(_tt.shape[1])
    assert (_tt[:, 2] == 97).all() and (_et[:, 2] == 97).all(), 'position 2 must always be the = token (id=p)'
    print('{}: p=97: {} train, {} test ({:.1%} train frac), seq_len={}'.format(
        _task, len(_tt), len(_et), len(_tt) / (len(_tt) + len(_et)), _tt.shape[1]))
    assert len(set(map(tuple, _tt.tolist())) & set(map(tuple, _et.tolist()))) == 0, 'train/test must not overlap'
    if _task == 'multiplication':
        assert (_tl != 0).all() and (_el != 0).all(), 'multiplication labels should never be 0 for prime p'
print('Dataset sanity checks passed for both tasks (canonical 3-token format confirmed).')


multiplication: p=97: 3686 train, 5530 test (40.0% train frac), seq_len=3
subtraction: p=97: 3763 train, 5646 test (40.0% train frac), seq_len=3
Dataset sanity checks passed for both tasks (canonical 3-token format confirmed).


## 5. Training / Inference

In [10]:
# Quick status check: how much of E5a/E5b is already checkpointed (from a prior session)?
# Safe to run before anything else -- just peeks at saved progress, does no computation.
for _name in ['E5a_multiplication', 'E5b_subtraction']:
    _ckpt = load_checkpoint(_name)
    if _ckpt is not None:
        print('  {:<20} {} runs already completed (from a previous session)'.format(_name, len(_ckpt['completed_keys'])))
    else:
        print('  {:<20} no checkpoint yet -- will start fresh'.format(_name))


  E5a_multiplication   no checkpoint yet -- will start fresh
  E5b_subtraction      no checkpoint yet -- will start fresh


In [11]:
@dataclass
class RunConfig:
    p: int
    seed: int
    task: str = 'addition'
    max_steps: int = 8000            # standard/baseline training length
    max_steps_safety_cap: int = 30000  # H8 fix: if grokked but not yet converged at max_steps,
    convergence_window: int = 3000     # keep training (checking a trailing weight-norm window) up
    lr: float = 1e-3                   # to this cap, instead of guessing a bigger fixed number
    weight_decay: float = 1.0
    d_model: int = 128
    n_heads: int = 4
    d_ff: int = 341
    train_frac: float = 0.4
    eval_every: int = 50
    fine_eval_horizon: int = 0       # H2 fix: check EVERY step up to this point instead of only
    fine_eval_every: int = 1         # every eval_every -- 0 = disabled. Not used by E5 (H2's
                                      # comparison only ever reads E1's data); left at default.
    track_full_history: bool = False  # record the full (step, train_acc, test_acc) trajectory --
                                       # not used by E5 (only E1's illustrative seed needs this).
    grok_threshold: float = 0.9
    pre_grok_test_ceiling: float = 0.3
    post_grok_offsets: tuple = (500, 1500, 4000)
    convergence_threshold: float = 0.05
    post_grok_dip_threshold: float = 0.5  # NEW: threshold below which a post-grok test-accuracy
                                            # reading counts as an "instability dip" -- discovered
                                            # by accident while fixing the pre_grok capture bug
                                            # (see below): test accuracy was found to fall this far
                                            # even long after initial grokking, in a substantial
                                            # fraction of runs. Tracked explicitly per-seed now.
    bit_widths: tuple = (2, 3, 4, 5, 6, 8, 10, 12, 16, 23)
    schemes: tuple = ('int', 'mantissa', 'nf')
    fine_grained_levels: tuple = (7, 8, 9, 10, 11, 12, 13, 14, 15)
    run_fine_grained: bool = True
    run_full_ptq_sweep: bool = True
    run_layerwise: bool = True
    run_layerwise_fine: bool = True
    layerwise_bit_widths: tuple = None
    layerwise_groups: tuple = None
    tag: str = ''


def _trend_based_converged(weight_norm_history, window_steps=6000, threshold=0.03):
    """H8 diagnostic, computed once at the end of training (not used for the live stopping
    decision -- see run_single_training below, which keeps the original, more conservative
    range-based check for that). The range-based check can't distinguish a weight norm that is
    genuinely still drifting from one that has settled into a bounded oscillation: a persistent
    oscillation never satisfies a narrow range threshold, no matter how many more steps you run.
    This instead fits a linear trend over the trailing window and checks whether the NET change is
    small relative to the current value -- tested directly against real trajectory data before
    being added here, and window_steps=6000 was the empirically best-performing choice tested."""
    if len(weight_norm_history) < 5:
        return None
    final_step = weight_norm_history[-1][0]
    recent = [(s, v) for s, v in weight_norm_history if s >= final_step - window_steps]
    if len(recent) < 5:
        return None
    steps_arr = np.array([s for s, v in recent], dtype=float)
    vals_arr = np.array([v for s, v in recent], dtype=float)
    steps_norm = steps_arr - steps_arr.mean()
    if steps_norm.std() < 1e-9:
        return None
    slope, intercept = np.polyfit(steps_norm, vals_arr, 1)
    total_change = slope * (steps_norm.max() - steps_norm.min())
    frac_trend = abs(total_change) / max(abs(vals_arr[-1]), 1e-9)
    return bool(frac_trend < threshold)


def run_single_training(cfg: RunConfig, data_cache: dict):
    """Train one model to (hopefully) full grokking on the canonical 3-token architecture, capture
    6 checkpoints, then run the requested subset of: the full PTQ sweep, the fine-grained level scan
    (H6), coarse + fine (H4/H4b) layer-group sensitivity, and the bin-width mechanism analysis with
    the init-checkpoint control (H3). Training runs to AT LEAST `max_steps`, then -- if grokked but
    not yet converged -- extends adaptively up to `max_steps_safety_cap`, checking a trailing
    weight-norm window rather than guessing a bigger fixed step count. `run_full_ptq_sweep=False` /
    `run_fine_grained=False` / `run_layerwise=False` let E5's calls below skip work that's confirmed
    unused downstream, keeping this secondary/replication experiment cheap."""
    set_seed(cfg.seed)
    key = (cfg.task, cfg.p, cfg.train_frac)
    if key not in data_cache:
        data_cache[key] = generate_dataset(cfg.task, cfg.p, cfg.train_frac, seed=0)
    train_tokens, train_labels, test_tokens, test_labels = data_cache[key]
    train_tokens, train_labels = train_tokens.to(DEVICE), train_labels.to(DEVICE)
    test_tokens, test_labels = test_tokens.to(DEVICE), test_labels.to(DEVICE)

    model = TinyTransformer(cfg.p, cfg.d_model, cfg.n_heads, cfg.d_ff, seq_len=3).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.98))

    def cpu_state_dict():
        return {k: v.detach().clone().cpu() for k, v in model.state_dict().items() if k in WEIGHT_PARAM_NAMES}

    def weight_norm():
        return math.sqrt(sum((p_ ** 2).sum().item() for n_, p_ in model.named_parameters() if n_ in WEIGHT_PARAM_NAMES))

    checkpoints = {}
    grok_step = None
    converged = False
    post_grok_dip_count = 0  # NEW: counts evaluated points after grok_step where test accuracy
                              # falls back below post_grok_dip_threshold -- discovered while
                              # fixing the pre_grok bug above; now tracked explicitly per seed.
    min_steps_after_grok = max(cfg.post_grok_offsets) + 500
    weight_norm_history = []
    train_test_history = [] if cfg.track_full_history else None

    model.eval()
    with torch.no_grad():
        init_train_acc = (model(train_tokens).argmax(-1) == train_labels).float().mean().item()
        init_test_acc = (model(test_tokens).argmax(-1) == test_labels).float().mean().item()
    checkpoints['init'] = (0, cpu_state_dict(), init_train_acc, init_test_acc, weight_norm())

    step = 0
    while True:
        model.train()
        logits = model(train_tokens)
        loss = F.cross_entropy(logits, train_labels)
        opt.zero_grad()
        loss.backward()
        opt.step()

        in_fine_window = cfg.fine_eval_horizon > 0 and step <= cfg.fine_eval_horizon
        should_eval = (
            step == cfg.max_steps
            or (in_fine_window and step % cfg.fine_eval_every == 0)
            or (not in_fine_window and step % cfg.eval_every == 0)
        )

        if should_eval:
            model.eval()
            with torch.no_grad():
                # Fixed: was reusing `logits` from this iteration's forward pass, computed BEFORE
                # opt.step() -- one step stale relative to the weights actually saved in this same
                # block. Confirmed via a bits=23 (documented no-op) sanity check that this caused
                # real disagreement, worst for pre_grok specifically (a rapidly-changing moment by
                # definition). Now computed fresh, after opt.step(), matching test_acc exactly.
                train_acc = (model(train_tokens).argmax(-1) == train_labels).float().mean().item()
                test_acc = (model(test_tokens).argmax(-1) == test_labels).float().mean().item()
            wn = weight_norm()
            weight_norm_history.append((step, wn))
            if train_test_history is not None:
                train_test_history.append((step, train_acc, test_acc))

            # CRITICAL FIX: previously this condition had no check on whether grokking had already
            # happened, only on whether pre_grok itself hadn't been captured yet. Confirmed via
            # direct inspection of E1's data that this let a LATE, post-grok test-accuracy dip get
            # mislabeled as "pre_grok" for every single seed that captured one (23/23) -- test
            # accuracy was found to oscillate well below this ceiling even long after initial
            # grokking, and since train accuracy stays high throughout, every dip satisfied the old
            # condition. Adding `grok_step is None` restores the intended meaning: pre_grok can now
            # only ever be captured chronologically before the first genuine grokking event. Applied
            # here for consistency with E1's fix, even though no current E5 analysis reads this
            # phase -- it's the same latent bug in the same shared training function.
            if 'pre_grok' not in checkpoints and grok_step is None and train_acc >= 0.99 and test_acc < cfg.pre_grok_test_ceiling:
                checkpoints['pre_grok'] = (step, cpu_state_dict(), train_acc, test_acc, wn)
            if grok_step is None and test_acc >= cfg.grok_threshold:
                grok_step = step
                checkpoints['just_grokked'] = (step, cpu_state_dict(), train_acc, test_acc, wn)
            if grok_step is not None:
                for offset in cfg.post_grok_offsets:
                    label = 'post_grok_{}'.format(offset)
                    if label not in checkpoints and step >= grok_step + offset:
                        checkpoints[label] = (step, cpu_state_dict(), train_acc, test_acc, wn)
                # NEW: quantify post-grok instability -- every evaluated point strictly after
                # grok_step where test accuracy falls back below post_grok_dip_threshold. Purely
                # additive counting; does not affect any existing checkpoint-capture logic above.
                if step > grok_step and test_acc < cfg.post_grok_dip_threshold:
                    post_grok_dip_count += 1

            if grok_step is not None and step >= cfg.max_steps and (step - grok_step) >= min_steps_after_grok:
                recent = [v for s, v in weight_norm_history if s >= step - cfg.convergence_window]
                if len(recent) >= 3:
                    frac_range = (max(recent) - min(recent)) / max(abs(recent[-1]), 1e-9)
                    if frac_range < cfg.convergence_threshold:
                        converged = True

        if step >= cfg.max_steps:
            if converged or grok_step is None or step >= cfg.max_steps_safety_cap:
                break
        step += 1

    final_step = step
    model.eval()
    with torch.no_grad():
        final_train_acc = (model(train_tokens).argmax(-1) == train_labels).float().mean().item()
        final_test_acc = (model(test_tokens).argmax(-1) == test_labels).float().mean().item()
    final_wn = weight_norm()
    checkpoints['cleaned_up'] = (final_step, cpu_state_dict(), final_train_acc, final_test_acc, final_wn)
    if not weight_norm_history or weight_norm_history[-1][0] != final_step:
        weight_norm_history.append((final_step, final_wn))
    if train_test_history is not None and (not train_test_history or train_test_history[-1][0] != final_step):
        train_test_history.append((final_step, final_train_acc, final_test_acc))
    grokked = grok_step is not None
    adaptive_converged = bool(converged) if grokked else None
    # NEW: trend-based diagnostic, computed once at the end using the full trajectory -- reported
    # alongside the original range-based adaptive_converged, not in place of it (see
    # _trend_based_converged's docstring for why both are worth keeping).
    adaptive_converged_trend = _trend_based_converged(weight_norm_history) if grokked else None

    plateau_info = None
    cleanup_converged = None
    reference_phase = 'post_grok_4000' if 'post_grok_4000' in checkpoints else ('post_grok_1500' if 'post_grok_1500' in checkpoints else None)
    if reference_phase is not None and 'cleaned_up' in checkpoints:
        wn_ref = checkpoints[reference_phase][4]
        wn_final = checkpoints['cleaned_up'][4]
        frac_change = abs(wn_final - wn_ref) / max(wn_ref, 1e-9)
        cleanup_converged = bool(frac_change < cfg.convergence_threshold)
        plateau_info = dict(weight_norm_reference=wn_ref, reference_phase=reference_phase,
                             weight_norm_cleaned_up=wn_final, frac_change=frac_change)

    eval_model = TinyTransformer(cfg.p, cfg.d_model, cfg.n_heads, cfg.d_ff, seq_len=3).to(DEVICE)

    ptq_rows = []
    if cfg.run_full_ptq_sweep:
        eval_phases = {k: v for k, v in checkpoints.items() if k != 'init'}
        for phase, (ckpt_step, sd, ck_train_acc, ck_test_acc, ck_wnorm) in eval_phases.items():
            sd_dev = {k: v.to(DEVICE) for k, v in sd.items()}
            for scheme in cfg.schemes:
                for bits in cfg.bit_widths:
                    q_sd = quantize_state_dict(sd_dev, bits, scheme=scheme)
                    tr_acc, te_acc = evaluate_state_dict(eval_model, q_sd, train_tokens, train_labels, test_tokens, test_labels)
                    ptq_rows.append(dict(p=cfg.p, seed=cfg.seed, task=cfg.task, phase=phase, phase_step=ckpt_step,
                                          fp32_train_acc=ck_train_acc, fp32_test_acc=ck_test_acc, weight_norm=ck_wnorm,
                                          scheme=scheme, bits=bits, q_train_acc=tr_acc, q_test_acc=te_acc,
                                          d_ff=cfg.d_ff, train_frac=cfg.train_frac, tag=cfg.tag))

    fine_rows = []
    if cfg.run_fine_grained and 'cleaned_up' in checkpoints:
        _, sd, ck_train_acc, ck_test_acc, _ = checkpoints['cleaned_up']
        sd_dev = {k: v.to(DEVICE) for k, v in sd.items()}
        for n_levels in cfg.fine_grained_levels:
            q_sd = {name: (int_quantize_nlevels(w, n_levels) if name in WEIGHT_PARAM_NAMES else w.clone())
                    for name, w in sd_dev.items()}
            tr_acc, te_acc = evaluate_state_dict(eval_model, q_sd, train_tokens, train_labels, test_tokens, test_labels)
            fine_rows.append(dict(p=cfg.p, seed=cfg.seed, task=cfg.task, n_levels=n_levels,
                                   fp32_train_acc=ck_train_acc, fp32_test_acc=ck_test_acc,
                                   q_train_acc=tr_acc, q_test_acc=te_acc, train_frac=cfg.train_frac, tag=cfg.tag))

    layerwise_rows = []
    layerwise_fine_rows = []
    if cfg.run_layerwise and 'cleaned_up' in checkpoints:
        _, sd, ck_train_acc, ck_test_acc, _ = checkpoints['cleaned_up']
        sd_dev = {k: v.to(DEVICE) for k, v in sd.items()}
        lw_bits = cfg.layerwise_bit_widths if cfg.layerwise_bit_widths is not None else cfg.bit_widths
        lw_groups = cfg.layerwise_groups if cfg.layerwise_groups is not None else list(LAYER_GROUPS.keys())
        for group_name in lw_groups:
            names = LAYER_GROUPS[group_name]
            for bits in lw_bits:
                q_sd = quantize_state_dict(sd_dev, bits, scheme='int', param_names=names)
                tr_acc, te_acc = evaluate_state_dict(eval_model, q_sd, train_tokens, train_labels, test_tokens, test_labels)
                layerwise_rows.append(dict(p=cfg.p, seed=cfg.seed, task=cfg.task, group=group_name, bits=bits,
                                            q_train_acc=tr_acc, q_test_acc=te_acc, d_ff=cfg.d_ff, tag=cfg.tag))
        if cfg.run_layerwise_fine:
            for group_name, names in LAYER_GROUPS_FINE_MLP.items():   # H4b
                for bits in lw_bits:
                    q_sd = quantize_state_dict(sd_dev, bits, scheme='int', param_names=names)
                    tr_acc, te_acc = evaluate_state_dict(eval_model, q_sd, train_tokens, train_labels, test_tokens, test_labels)
                    layerwise_fine_rows.append(dict(p=cfg.p, seed=cfg.seed, task=cfg.task, group=group_name, bits=bits,
                                                     q_train_acc=tr_acc, q_test_acc=te_acc, d_ff=cfg.d_ff, tag=cfg.tag))

    mechanism_row = None
    if 'just_grokked' in checkpoints and 'cleaned_up' in checkpoints and 'init' in checkpoints:
        _, sd_grok, *_ = checkpoints['just_grokked']
        _, sd_clean, *_ = checkpoints['cleaned_up']
        _, sd_init, *_ = checkpoints['init']
        erasure_cleanup_by_bits, erasure_fulltrain_by_bits = {}, {}
        for bits in cfg.bit_widths:
            fracs_cleanup, fracs_fulltrain = [], []
            for name in WEIGHT_PARAM_NAMES:
                delta_cleanup = (sd_clean[name] - sd_grok[name]).abs()
                delta_fulltrain = (sd_clean[name] - sd_init[name]).abs()
                if bits >= 23:
                    fracs_cleanup.append(0.0); fracs_fulltrain.append(0.0)
                    continue
                qmax = max(2 ** (bits - 1) - 1, 1)
                scale = sd_clean[name].abs().max().item() / qmax
                if scale < 1e-12:
                    fracs_cleanup.append(0.0); fracs_fulltrain.append(0.0)
                else:
                    fracs_cleanup.append((delta_cleanup < scale).float().mean().item())
                    fracs_fulltrain.append((delta_fulltrain < scale).float().mean().item())
            erasure_cleanup_by_bits[bits] = float(np.mean(fracs_cleanup))
            erasure_fulltrain_by_bits[bits] = float(np.mean(fracs_fulltrain))
        mechanism_row = dict(p=cfg.p, seed=cfg.seed, task=cfg.task, tag=cfg.tag,
                              grok_step=grok_step, cleanup_steps=final_step - grok_step,
                              erasure_cleanup_by_bits=erasure_cleanup_by_bits,
                              erasure_fulltrain_by_bits=erasure_fulltrain_by_bits)

    summary = dict(p=cfg.p, seed=cfg.seed, task=cfg.task, tag=cfg.tag, grokked=grokked, grok_step=grok_step,
                    n_checkpoints=len(checkpoints), checkpoint_phases=sorted(checkpoints.keys()),
                    final_train_acc=final_train_acc, final_test_acc=final_test_acc,
                    d_ff=cfg.d_ff, train_frac=cfg.train_frac, plateau_info=plateau_info,
                    cleanup_converged=cleanup_converged,
                    actual_steps_used=final_step, adaptive_converged=adaptive_converged,
                    adaptive_converged_trend=adaptive_converged_trend,
                    post_grok_dip_count=post_grok_dip_count)

    trajectory_info = dict(p=cfg.p, seed=cfg.seed, task=cfg.task, tag=cfg.tag,
                            weight_norm_history=weight_norm_history, train_test_history=train_test_history)

    return summary, ptq_rows, layerwise_rows, layerwise_fine_rows, mechanism_row, fine_rows, trajectory_info


In [12]:
# ---- Global experiment configuration: edit here to scale up/down ----
QUICK_TEST = False   # set True for a fast (~few minutes) pipeline sanity check before the full run

if QUICK_TEST:
    # p=37 (wd=5.0, train_frac=0.6) reliably groks by ~step 300.
    P_PRIMARY, P_SECONDARY = 37, 29
    N_SEEDS_MULT = 1
    N_SEEDS_SUB = 1
    MAX_STEPS = 900
    WEIGHT_DECAY = 5.0
    TRAIN_FRAC = 0.6
    BIT_WIDTHS = (2, 4, 8, 23)
    POST_GROK_OFFSETS = (100, 300)  # smaller than the full-scale (500,1500,4000) -- H8's adaptive
                                      # stopping waits for max(post_grok_offsets)+500 before it can
                                      # even check convergence, so QUICK_TEST needs small offsets
    MAX_STEPS_SAFETY_CAP = 3000       # without this, a grokked-but-not-yet-converged run could
                                      # silently extend toward the full-scale 30000-step cap
    CONVERGENCE_WINDOW = 300          # the full-scale 3000-step window would span almost this
                                      # entire run, making convergence very hard to detect
else:
    P_PRIMARY, P_SECONDARY = 97, 59
    N_SEEDS_MULT = 25
    N_SEEDS_SUB = 25
    MAX_STEPS = 12000
    WEIGHT_DECAY = 1.0
    TRAIN_FRAC = 0.4
    BIT_WIDTHS = (2, 3, 4, 5, 6, 8, 10, 12, 16, 23)
    POST_GROK_OFFSETS = (500, 1500, 4000)
    MAX_STEPS_SAFETY_CAP = 30000
    CONVERGENCE_WINDOW = 3000

SCHEMES = ('int', 'mantissa', 'nf')
TIME_BUDGET_SEC = 36.0 * 3600  # generous; per-run cost empirically ~30-150s/run on a T4

data_cache = {}

def time_remaining():
    return TIME_BUDGET_SEC - (time.time() - EXPERIMENT_START_TIME)

print('Global config: P_PRIMARY={}, P_SECONDARY={}, MAX_STEPS={}, WEIGHT_DECAY={}, TRAIN_FRAC={}, '
      'N_SEEDS_MULT={}, N_SEEDS_SUB={}, QUICK_TEST={}'.format(
    P_PRIMARY, P_SECONDARY, MAX_STEPS, WEIGHT_DECAY, TRAIN_FRAC, N_SEEDS_MULT, N_SEEDS_SUB, QUICK_TEST))


Global config: P_PRIMARY=97, P_SECONDARY=59, MAX_STEPS=12000, WEIGHT_DECAY=1.0, TRAIN_FRAC=0.4, N_SEEDS_MULT=25, N_SEEDS_SUB=25, QUICK_TEST=False


## 6. E5: Cross-Task Replication (Multiplication, Subtraction)

Checkpointed independently for each task: progress is saved after every completed run, so a
crashed/disconnected session resumes here instead of restarting. `run_layerwise=False`,
`run_fine_grained=False` for both: confirmed and removed, since H5's cross-task analysis only ever
reads PTQ rows downstream.


In [13]:
mult_summaries, mult_ptq_rows, mult_layerwise_rows = [], [], []
mult_layerwise_fine_rows, mult_mechanism_rows, mult_fine_rows, mult_trajectory_info = [], [], [], []

def _e5a_run(combo):
    cfg = RunConfig(p=combo['p'], seed=combo['seed'], task='multiplication', max_steps=MAX_STEPS, bit_widths=BIT_WIDTHS,
                     schemes=SCHEMES, run_fine_grained=False, run_layerwise=False,
                     train_frac=TRAIN_FRAC, weight_decay=WEIGHT_DECAY, tag='E5a_multiplication',
                     post_grok_offsets=POST_GROK_OFFSETS, max_steps_safety_cap=MAX_STEPS_SAFETY_CAP,
                     convergence_window=CONVERGENCE_WINDOW)
    t0 = time.time()
    result = run_single_training(cfg, data_cache)
    summary = result[0]
    print('  p={:>2} seed={} grokked={} grok_step={} final_test_acc={:.3f} steps_used={} dip_count={} conv(range/trend)={}/{} ({:.1f}s)'.format(
        combo['p'], combo['seed'], summary['grokked'], summary['grok_step'], summary['final_test_acc'],
        summary['actual_steps_used'], summary['post_grok_dip_count'], summary['adaptive_converged'],
        summary['adaptive_converged_trend'], time.time() - t0))
    return result

e5a_combos = [{'p': p, 'seed': s} for p in [P_PRIMARY, P_SECONDARY] for s in range(N_SEEDS_MULT)]
print('[E5a / multiplication] p in [{}, {}], seeds={}, max_steps={}'.format(P_PRIMARY, P_SECONDARY, N_SEEDS_MULT, MAX_STEPS))
e5a_out = checkpointed_sweep('E5a_multiplication', e5a_combos, ['p', 'seed'], _e5a_run,
                              ['summaries', 'ptq_rows', 'layerwise_rows', 'layerwise_fine_rows', 'mechanism_rows', 'fine_rows', 'trajectory_info'])
mult_summaries = e5a_out['summaries']
mult_ptq_rows = e5a_out['ptq_rows']
mult_layerwise_rows = e5a_out['layerwise_rows']
mult_layerwise_fine_rows = e5a_out['layerwise_fine_rows']
mult_mechanism_rows = e5a_out['mechanism_rows']
mult_fine_rows = e5a_out['fine_rows']
mult_trajectory_info = e5a_out['trajectory_info']

print()
print('E5a done. Total elapsed: {:.1f} min'.format((time.time() - EXPERIMENT_START_TIME) / 60))
print('Grokked: {} / {} runs'.format(sum(1 for s in mult_summaries if s['grokked']), len(mult_summaries)))


[E5a / multiplication] p in [97, 59], seeds=25, max_steps=12000
[E5a_multiplication] Starting fresh: 50 runs planned.
  p=97 seed=0 grokked=True grok_step=650 final_test_acc=0.992 steps_used=30000 dip_count=0 conv(range/trend)=False/True (281.2s)
  p=97 seed=1 grokked=True grok_step=550 final_test_acc=0.997 steps_used=27950 dip_count=9 conv(range/trend)=True/True (260.5s)
  p=97 seed=2 grokked=True grok_step=600 final_test_acc=0.998 steps_used=30000 dip_count=4 conv(range/trend)=False/True (278.7s)
  p=97 seed=3 grokked=True grok_step=3550 final_test_acc=0.984 steps_used=30000 dip_count=0 conv(range/trend)=False/False (278.8s)
  p=97 seed=4 grokked=True grok_step=4950 final_test_acc=0.869 steps_used=12150 dip_count=4 conv(range/trend)=True/False (113.9s)
  p=97 seed=5 grokked=True grok_step=900 final_test_acc=0.980 steps_used=26200 dip_count=12 conv(range/trend)=True/False (243.8s)
  p=97 seed=6 grokked=True grok_step=750 final_test_acc=0.994 steps_used=30000 dip_count=3 conv(range/tre

In [14]:
sub_summaries, sub_ptq_rows, sub_layerwise_rows = [], [], []
sub_layerwise_fine_rows, sub_mechanism_rows, sub_fine_rows, sub_trajectory_info = [], [], [], []

def _e5b_run(combo):
    cfg = RunConfig(p=combo['p'], seed=combo['seed'], task='subtraction', max_steps=MAX_STEPS, bit_widths=BIT_WIDTHS,
                     schemes=SCHEMES, run_fine_grained=False, run_layerwise=False,
                     train_frac=TRAIN_FRAC, weight_decay=WEIGHT_DECAY, tag='E5b_subtraction',
                     post_grok_offsets=POST_GROK_OFFSETS, max_steps_safety_cap=MAX_STEPS_SAFETY_CAP,
                     convergence_window=CONVERGENCE_WINDOW)
    t0 = time.time()
    result = run_single_training(cfg, data_cache)
    summary = result[0]
    print('  p={:>2} seed={} grokked={} grok_step={} final_test_acc={:.3f} steps_used={} dip_count={} conv(range/trend)={}/{} ({:.1f}s)'.format(
        combo['p'], combo['seed'], summary['grokked'], summary['grok_step'], summary['final_test_acc'],
        summary['actual_steps_used'], summary['post_grok_dip_count'], summary['adaptive_converged'],
        summary['adaptive_converged_trend'], time.time() - t0))
    return result

e5b_combos = [{'p': p, 'seed': s} for p in [P_PRIMARY, P_SECONDARY] for s in range(N_SEEDS_SUB)]
print('[E5b / subtraction] p in [{}, {}], seeds={}, max_steps={}'.format(P_PRIMARY, P_SECONDARY, N_SEEDS_SUB, MAX_STEPS))
e5b_out = checkpointed_sweep('E5b_subtraction', e5b_combos, ['p', 'seed'], _e5b_run,
                              ['summaries', 'ptq_rows', 'layerwise_rows', 'layerwise_fine_rows', 'mechanism_rows', 'fine_rows', 'trajectory_info'])
sub_summaries = e5b_out['summaries']
sub_ptq_rows = e5b_out['ptq_rows']
sub_layerwise_rows = e5b_out['layerwise_rows']
sub_layerwise_fine_rows = e5b_out['layerwise_fine_rows']
sub_mechanism_rows = e5b_out['mechanism_rows']
sub_fine_rows = e5b_out['fine_rows']
sub_trajectory_info = e5b_out['trajectory_info']

print()
print('E5b done. Total elapsed: {:.1f} min'.format((time.time() - EXPERIMENT_START_TIME) / 60))
print('Grokked: {} / {} runs'.format(sum(1 for s in sub_summaries if s['grokked']), len(sub_summaries)))


[E5b / subtraction] p in [97, 59], seeds=25, max_steps=12000
[E5b_subtraction] Starting fresh: 50 runs planned.
  p=97 seed=0 grokked=False grok_step=None final_test_acc=0.261 steps_used=12000 dip_count=0 conv(range/trend)=None/None (114.4s)
  p=97 seed=1 grokked=True grok_step=7150 final_test_acc=0.609 steps_used=13000 dip_count=2 conv(range/trend)=True/False (125.4s)
  p=97 seed=2 grokked=True grok_step=4450 final_test_acc=0.964 steps_used=19600 dip_count=2 conv(range/trend)=True/False (187.5s)
  p=97 seed=3 grokked=True grok_step=4600 final_test_acc=1.000 steps_used=30000 dip_count=4 conv(range/trend)=False/False (286.2s)
  p=97 seed=4 grokked=True grok_step=5850 final_test_acc=0.987 steps_used=17400 dip_count=3 conv(range/trend)=True/False (167.0s)
  p=97 seed=5 grokked=False grok_step=None final_test_acc=0.029 steps_used=12000 dip_count=0 conv(range/trend)=None/None (114.7s)
  p=97 seed=6 grokked=False grok_step=None final_test_acc=0.287 steps_used=12000 dip_count=0 conv(range/tre

## 7. Export Dataset

This notebook's only job is to produce E5's complete raw dataset (both multiplication and
subtraction) for the downstream analysis notebook (which will run E1/E6/E7, all statistics, and
all figures) to load without recomputing anything. Every output is saved in a form that preserves
nested fields -- `plateau_info`, `checkpoint_phases`, `trajectory_info` -- that a naive `.to_csv()`
would silently drop: summaries and trajectory data go to JSON, every already-flat table goes to
CSV. `layerwise_rows`, `layerwise_fine_rows`, and `fine_rows` are expected to be empty for both
tasks (`run_layerwise=False`, `run_fine_grained=False`) -- the export step skips saving empty files
rather than cluttering the output. `mechanism_rows` is a separate case: it's cheap tensor
arithmetic, not gated by either flag, so it will be non-empty for any seed that grokked -- saved
here for completeness even though the downstream H3 analysis only reads E1's mechanism data.


In [15]:
os.makedirs(os.path.join(OUTPUT_ROOT, 'logs'), exist_ok=True)


def mechanism_rows_to_long_df(mechanism_rows):
    long_rows = []
    for row in mechanism_rows:
        for bits in row['erasure_cleanup_by_bits']:
            long_rows.append(dict(p=row['p'], seed=row['seed'], task=row['task'], tag=row['tag'],
                                   grok_step=row['grok_step'], cleanup_steps=row['cleanup_steps'], bits=bits,
                                   erasure_cleanup=row['erasure_cleanup_by_bits'][bits],
                                   erasure_fulltrain=row['erasure_fulltrain_by_bits'][bits]))
    return pd.DataFrame(long_rows)


def _save_csv(rows, name):
    if rows:
        pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_ROOT, 'logs', name), index=False)
        print('Saved: {} ({} rows)'.format(name, len(rows)))
    else:
        print('Skipped (empty -- expected for E5: run_layerwise=False, run_fine_grained=False, or QUICK_TEST): {}'.format(name))


def _traj_to_jsonable(traj_list):
    out = []
    for t in traj_list:
        t2 = dict(t)
        t2['weight_norm_history'] = [list(x) for x in t2['weight_norm_history']]
        if t2['train_test_history'] is not None:
            t2['train_test_history'] = [list(x) for x in t2['train_test_history']]
        out.append(t2)
    return out


def export_task(prefix, summaries, ptq_rows, layerwise_rows, layerwise_fine_rows, mechanism_rows, fine_rows, trajectory_info):
    with open(os.path.join(OUTPUT_ROOT, 'logs', '{}_summaries.json'.format(prefix)), 'w') as f:
        json.dump(summaries, f, indent=1)
    print('Saved: {}_summaries.json ({} runs)'.format(prefix, len(summaries)))

    _save_csv(ptq_rows, '{}_ptq_rows.csv'.format(prefix))
    _save_csv(layerwise_rows, '{}_layerwise_rows.csv'.format(prefix))
    _save_csv(layerwise_fine_rows, '{}_layerwise_fine_rows.csv'.format(prefix))
    _save_csv(fine_rows, '{}_fine_rows.csv'.format(prefix))

    df_mech = mechanism_rows_to_long_df(mechanism_rows)
    if not df_mech.empty:
        df_mech.to_csv(os.path.join(OUTPUT_ROOT, 'logs', '{}_mechanism_rows.csv'.format(prefix)), index=False)
        print('Saved: {}_mechanism_rows.csv ({} rows)'.format(prefix, len(df_mech)))
    else:
        print('Skipped (empty -- no grokked runs, or QUICK_TEST): {}_mechanism_rows.csv'.format(prefix))

    with open(os.path.join(OUTPUT_ROOT, 'logs', '{}_trajectory_info.json'.format(prefix)), 'w') as f:
        json.dump(_traj_to_jsonable(trajectory_info), f, indent=1)
    print('Saved: {}_trajectory_info.json ({} runs)'.format(prefix, len(trajectory_info)))


print('--- Multiplication ---')
export_task('e5a_multiplication', mult_summaries, mult_ptq_rows, mult_layerwise_rows,
            mult_layerwise_fine_rows, mult_mechanism_rows, mult_fine_rows, mult_trajectory_info)
print('--- Subtraction ---')
export_task('e5b_subtraction', sub_summaries, sub_ptq_rows, sub_layerwise_rows,
            sub_layerwise_fine_rows, sub_mechanism_rows, sub_fine_rows, sub_trajectory_info)

# Environment/config record, matching the E1 notebook's pattern -- documents exactly what produced
# this dataset (architecture, hyperparameters, package versions), for reproducibility.
env_info = {
    'python_version': sys.version, 'torch_version': torch.__version__, 'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'numpy_version': np.__version__, 'pandas_version': pd.__version__, 'base_seed': BASE_SEED,
    'runtime_env': RUNTIME_ENV, 'output_root': OUTPUT_ROOT, 'zip_path': ZIP_PATH,
    'quick_test': QUICK_TEST, 'p_primary': P_PRIMARY, 'p_secondary': P_SECONDARY,
    'n_seeds_mult': N_SEEDS_MULT, 'n_seeds_sub': N_SEEDS_SUB, 'max_steps': MAX_STEPS,
    'max_steps_safety_cap': MAX_STEPS_SAFETY_CAP, 'weight_decay': WEIGHT_DECAY, 'train_frac': TRAIN_FRAC,
    'bit_widths': list(BIT_WIDTHS), 'schemes': list(SCHEMES), 'post_grok_offsets': list(POST_GROK_OFFSETS),
    'convergence_window': CONVERGENCE_WINDOW, 'run_layerwise': False, 'run_fine_grained': False,
    'total_runtime_minutes': (time.time() - EXPERIMENT_START_TIME) / 60,
}
with open(os.path.join(OUTPUT_ROOT, 'logs', 'e5_environment_and_config.json'), 'w') as f:
    json.dump(env_info, f, indent=1)
print('Saved: e5_environment_and_config.json')


--- Multiplication ---
Saved: e5a_multiplication_summaries.json (50 runs)
Saved: e5a_multiplication_ptq_rows.csv (7290 rows)
Skipped (empty -- expected for E5: run_layerwise=False, run_fine_grained=False, or QUICK_TEST): e5a_multiplication_layerwise_rows.csv
Skipped (empty -- expected for E5: run_layerwise=False, run_fine_grained=False, or QUICK_TEST): e5a_multiplication_layerwise_fine_rows.csv
Skipped (empty -- expected for E5: run_layerwise=False, run_fine_grained=False, or QUICK_TEST): e5a_multiplication_fine_rows.csv
Saved: e5a_multiplication_mechanism_rows.csv (470 rows)
Saved: e5a_multiplication_trajectory_info.json (50 runs)
--- Subtraction ---
Saved: e5b_subtraction_summaries.json (50 runs)
Saved: e5b_subtraction_ptq_rows.csv (4920 rows)
Skipped (empty -- expected for E5: run_layerwise=False, run_fine_grained=False, or QUICK_TEST): e5b_subtraction_layerwise_rows.csv
Skipped (empty -- expected for E5: run_layerwise=False, run_fine_grained=False, or QUICK_TEST): e5b_subtraction_l

### Verify the export round-trips correctly

A quick sanity check that every saved file reloads to the same shape it was saved with -- catches
a serialization bug here rather than silently in the downstream notebook.


In [16]:
for _prefix, _summaries, _ptq in [('e5a_multiplication', mult_summaries, mult_ptq_rows),
                                    ('e5b_subtraction', sub_summaries, sub_ptq_rows)]:
    _summ_path = os.path.join(OUTPUT_ROOT, 'logs', '{}_summaries.json'.format(_prefix))
    with open(_summ_path) as f:
        _check = json.load(f)
    assert len(_check) == len(_summaries), '{} summaries count mismatch after reload'.format(_prefix)
    if _check:
        assert _check[0]['p'] == _summaries[0]['p'], '{} summaries content mismatch after reload'.format(_prefix)
    print('{}_summaries.json round-trips correctly ({} runs).'.format(_prefix, len(_check)))

    if _ptq:
        _ptq_path = os.path.join(OUTPUT_ROOT, 'logs', '{}_ptq_rows.csv'.format(_prefix))
        _check_ptq = pd.read_csv(_ptq_path)
        assert len(_check_ptq) == len(_ptq), '{} ptq_rows count mismatch after reload'.format(_prefix)
        print('{}_ptq_rows.csv round-trips correctly ({} rows).'.format(_prefix, len(_check_ptq)))


e5a_multiplication_summaries.json round-trips correctly (50 runs).
e5a_multiplication_ptq_rows.csv round-trips correctly (7290 rows).
e5b_subtraction_summaries.json round-trips correctly (50 runs).
e5b_subtraction_ptq_rows.csv round-trips correctly (4920 rows).


## 8. Package and Download

In [17]:
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_ROOT):
        for file in files:
            full = os.path.join(root, file)
            zf.write(full, arcname=os.path.relpath(full, os.path.dirname(OUTPUT_ROOT)))
print('Created {} ({:.1f} KB)'.format(ZIP_PATH, os.path.getsize(ZIP_PATH) / 1024))


Created /kaggle/working/e5_dataset.zip (708.0 KB)


In [18]:
# ============ FINAL CELL: summary, file listing, zip confirmation, download ============
print('=' * 70)
print('E5 DATASET EXPORT SUMMARY')
print('=' * 70)
print('Total runtime: {:.1f} minutes'.format((time.time() - EXPERIMENT_START_TIME) / 60))
print('Multiplication: {} runs ({} grokked)'.format(len(mult_summaries), sum(1 for s in mult_summaries if s['grokked'])))
print('Subtraction:    {} runs ({} grokked)'.format(len(sub_summaries), sum(1 for s in sub_summaries if s['grokked'])))
print()


def _report_diagnostics(task_label, summaries, ptq_rows):
    print('--- {}: pre_grok fix verification ---'.format(task_label))
    if ptq_rows:
        df_check = pd.DataFrame(ptq_rows)[['p', 'seed', 'phase', 'phase_step']].drop_duplicates()
        pivot_check = df_check.pivot_table(index=['p', 'seed'], columns='phase', values='phase_step', aggfunc='first')
        if 'pre_grok' in pivot_check.columns and 'just_grokked' in pivot_check.columns:
            both_check = pivot_check.dropna(subset=['pre_grok', 'just_grokked'])
            n_correct = (both_check['pre_grok'] < both_check['just_grokked']).sum()
            n_wrong = (both_check['pre_grok'] > both_check['just_grokked']).sum()
            print('Seeds with both pre_grok and just_grokked captured: {}'.format(len(both_check)))
            print('  pre_grok correctly BEFORE just_grokked: {}'.format(n_correct))
            print('  pre_grok WRONGLY after just_grokked (should be 0): {}'.format(n_wrong))
            if n_wrong > 0:
                print('  *** FIX DID NOT FULLY WORK -- investigate before trusting this data ***')
            elif n_correct == 0:
                print('  No seeds captured pre_grok chronologically correctly -- not necessarily a')
                print('  sign anything is broken (E5 does not currently use this phase for any')
                print('  analysis; this check is here purely for consistency with E1).')
            else:
                print('  Fix confirmed working: every captured pre_grok checkpoint is chronologically valid.')
        else:
            print('pre_grok phase not present in this run -- nothing to verify yet.')
    else:
        print('No PTQ rows to check yet.')
    print()

    print('--- {}: H8 convergence diagnostics ---'.format(task_label))
    grokked_summaries = [s for s in summaries if s['grokked']]
    if grokked_summaries:
        range_conv = [s['adaptive_converged'] for s in grokked_summaries if s['adaptive_converged'] is not None]
        trend_conv = [s['adaptive_converged_trend'] for s in grokked_summaries if s['adaptive_converged_trend'] is not None]
        if range_conv:
            print('Range-based converged:  {}/{} = {:.1%}'.format(sum(range_conv), len(range_conv), np.mean(range_conv)))
        if trend_conv:
            print('Trend-based converged:  {}/{} = {:.1%}'.format(sum(trend_conv), len(trend_conv), np.mean(trend_conv)))
        dip_counts = [s['post_grok_dip_count'] for s in grokked_summaries]
        print('Post-grok instability dips (test_acc < {}), across {} grokked runs:'.format(
            RunConfig.__dataclass_fields__['post_grok_dip_threshold'].default, len(dip_counts)))
        print('  mean={:.1f}  median={:.0f}  max={}  seeds with 0 dips={}'.format(
            np.mean(dip_counts), np.median(dip_counts), max(dip_counts), sum(1 for d in dip_counts if d == 0)))
    else:
        print('No grokked runs yet.')
    print()


_report_diagnostics('Multiplication', mult_summaries, mult_ptq_rows)
_report_diagnostics('Subtraction', sub_summaries, sub_ptq_rows)

print('Generated files:')
for root, _, files in os.walk(OUTPUT_ROOT):
    for file in sorted(files):
        print(' ', os.path.join(root, file))
print()
assert os.path.exists(ZIP_PATH), '{} was not created!'.format(ZIP_PATH)
zip_size_kb = os.path.getsize(ZIP_PATH) / 1024
print('{} confirmed on disk ({:.1f} KB).'.format(ZIP_PATH, zip_size_kb))
print('Runtime environment: {}'.format(RUNTIME_ENV))
print('=' * 70)
print()
print('Load this dataset in the downstream (E1/E6/E7 + evaluation + figures) notebook with:')
print("  with open('e5a_multiplication_summaries.json') as f: mult_summaries = json.load(f)")
print("  mult_ptq_rows = pd.read_csv('e5a_multiplication_ptq_rows.csv').to_dict('records')")
print('  (etc. for e5b_subtraction_* and the other files listed above)')
print('=' * 70)

if RUNTIME_ENV == 'colab':
    try:
        from google.colab import files as _colab_files
        _colab_files.download(ZIP_PATH)
        print('Download triggered.')
    except Exception as e:
        print('Could not auto-trigger download ({}). Find your results at: {}'.format(e, ZIP_PATH))

else:
    # Embeds the file bytes directly as a base64 data URI -- the browser decodes it client-side on
    # click, so there is no server-side file path to resolve and therefore nothing that can 404.
    # Works the same way in Kaggle, local Jupyter, JupyterLab, and the VS Code notebook viewer.
    MAX_INLINE_MB = 25
    if zip_size_kb / 1024 <= MAX_INLINE_MB:
        try:
            import base64
            from IPython.display import HTML, display
            with open(ZIP_PATH, 'rb') as f:
                b64_data = base64.b64encode(f.read()).decode('ascii')
            html = (
                '<a download="e5_dataset.zip" href="data:application/zip;base64,{}" '
                'style="display:inline-block;padding:8px 16px;background:#1b4f72;color:white;'
                'border-radius:4px;text-decoration:none;font-family:sans-serif;">'
                'Click to download e5_dataset.zip ({:.1f} KB)</a>'
            ).format(b64_data, zip_size_kb)
            display(HTML(html))
            print('Click the button above -- this downloads directly from the embedded file data,')
            print('so it works regardless of the notebook server file-serving configuration.')
        except Exception as e:
            print('Could not render an inline download button ({}).'.format(e))
            print('Find your results at: {}'.format(ZIP_PATH))
    else:
        print('e5_dataset.zip is {:.1f} MB -- too large to embed inline reliably.'.format(zip_size_kb / 1024))
        print('Find your results at: {}, or use the notebook file browser'.format(ZIP_PATH))
        print('(the sidebar in JupyterLab or Kaggle) to navigate there and download it directly.')

    if RUNTIME_ENV == 'kaggle':
        print()
        print('On Kaggle specifically: the button above may or may not render depending on Kaggle')
        print('output sandboxing. If not, the reliable path is the Output pane in the right sidebar')
        print('(or the Output tab after committing this notebook) -- click e5_dataset.zip there.')


E5 DATASET EXPORT SUMMARY
Total runtime: 252.9 minutes
Multiplication: 50 runs (47 grokked)
Subtraction:    50 runs (16 grokked)

--- Multiplication: pre_grok fix verification ---
Seeds with both pre_grok and just_grokked captured: 4
  pre_grok correctly BEFORE just_grokked: 4
  pre_grok WRONGLY after just_grokked (should be 0): 0
  Fix confirmed working: every captured pre_grok checkpoint is chronologically valid.

--- Multiplication: H8 convergence diagnostics ---
Range-based converged:  5/47 = 10.6%
Trend-based converged:  18/47 = 38.3%
Post-grok instability dips (test_acc < 0.5), across 47 grokked runs:
  mean=2.1  median=1  max=12  seeds with 0 dips=14

--- Subtraction: pre_grok fix verification ---
Seeds with both pre_grok and just_grokked captured: 16
  pre_grok correctly BEFORE just_grokked: 16
  pre_grok WRONGLY after just_grokked (should be 0): 0
  Fix confirmed working: every captured pre_grok checkpoint is chronologically valid.

--- Subtraction: H8 convergence diagnostics 

Click the button above -- this downloads directly from the embedded file data,
so it works regardless of the notebook server file-serving configuration.

On Kaggle specifically: the button above may or may not render depending on Kaggle
output sandboxing. If not, the reliable path is the Output pane in the right sidebar
(or the Output tab after committing this notebook) -- click e5_dataset.zip there.
